# SAE Biological Alignment Analysis

This notebook reframes the structure-alignment study as a **multi-layer comparative analysis** across SAE layers 0, 5, and 11.

## Goals
1. Load the three trained SAE checkpoints in a consistent, configurable way.
2. Recompute or reuse cached nucleotide-level structure alignment statistics from bpRNA-90.
3. Compare how structure-selective features differ across layers.
4. Produce publication-ready figures with bold, A4-friendly typography.
5. Keep figure export optional: every PDF export line is present but commented.


## Usage Notes

- Set `DEVICE_PREFERENCE` to `auto`, `cuda`, `mps`, or `cpu`; `auto` selects the best available backend.
- It does **not** execute automatically here.
- Cached intermediates are stored under `.cache/analysis/structure_analysis/...`.
- If you want to regenerate figures as PDF, uncomment the `fig.savefig(...)` line in the corresponding plot cell.


In [ ]:
from pathlib import Path
import gc
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score

from src.notebook_analysis_utils import (
    STRUCTURE_LABELS,
    STRUCTURE_PALETTE,
    auto_discover_latest_layer_paths,
    beautify_axes,
    collect_structure_alignment,
    compute_structure_feature_statistics,
    load_layer_catalog,
    load_layer_model,
    resolve_device,
    safe_unload,
    set_publication_style,
    sha1_digest,
    stream_bprna_dataframe,
    summarize_structure_distribution,
)


def show_and_close(fig, stem: str | None = None) -> None:
    if stem is not None:
        figure_dir = globals().get("FIGURE_DIR")
        if figure_dir is None:
            raise RuntimeError("FIGURE_DIR must be configured before saving figures.")
        fig.savefig(figure_dir / f"{stem}.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    gc.collect()


def annotate_panel(ax, label: str) -> None:
    """Use the top margin for a compact, consistently aligned layer badge."""
    ax.set_title(
        label,
        loc="right",
        fontsize=12,
        fontweight="bold",
        pad=8,
        color="#374151",
        bbox={
            "boxstyle": "round,pad=0.22",
            "facecolor": "#f3f4f6",
            "edgecolor": "#9ca3af",
            "linewidth": 0.8,
        },
    )


## Configuration

The notebook defaults to the current three production checkpoints. If newer runs exist, set `AUTO_DISCOVER_LATEST = True` to substitute the most recent run per layer automatically.


In [ ]:
PROJECT_ROOT = Path.cwd()
ARCHIVE_ROOT = PROJECT_ROOT.parent if (PROJECT_ROOT.parent / "models").is_dir() else PROJECT_ROOT
CHECKPOINT_ROOT = ARCHIVE_ROOT / "models"

DEFAULT_LAYER_PATHS = {
    0: CHECKPOINT_ROOT / "layer_00",
    5: CHECKPOINT_ROOT / "layer_05",
    11: CHECKPOINT_ROOT / "layer_11",
}
AUTO_DISCOVER_LATEST = False
DISCOVERED_LAYER_PATHS = auto_discover_latest_layer_paths(CHECKPOINT_ROOT)
LAYER_PATHS = {
    layer_index: DISCOVERED_LAYER_PATHS.get(layer_index, default_path)
    for layer_index, default_path in DEFAULT_LAYER_PATHS.items()
} if AUTO_DISCOVER_LATEST else DEFAULT_LAYER_PATHS

REFERENCE_LAYER = 0
RUNTIME_TARGET = os.environ.get("SPIRAL_RUNTIME_TARGET", "local")
DEVICE_PREFERENCE = os.environ.get("SPIRAL_DEVICE", "auto")
DEVICE = resolve_device(DEVICE_PREFERENCE)
MODEL_QUANTIZATION = None
USE_CACHE = True
REQUIRE_CACHE = True
CACHE_SCHEMA_VERSION = 2

BPRNA_DATASET_ID = "multimolecule/bprna-90"
BPRNA_SPLIT = "train"
NUM_SEQUENCES = 25_000
MAX_SEQ_LENGTH = 512
NUC_RESERVOIR_SIZE = 300_000
MIN_SEQUENCE_COUNT = 5
TOP_FEATURES_PER_LAYER = 10
ACTIVATION_THRESHOLDS = [0.0, 0.05, 0.1, 0.25, 0.5, 1.0]
PCA_SAMPLE_SIZE = 4_500
PCA_ACTIVE_FEATURE_CAP = 256
PCA_RANDOM_STATE = 42

LAYER_COLORS = {
    0: "#0f766e",
    5: "#b45309",
    11: "#7c3aed",
}

ANALYSIS_KEY = sha1_digest(
    {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "layer_paths": {key: str(value) for key, value in LAYER_PATHS.items()},
        "model_quantization": MODEL_QUANTIZATION or "checkpoint-default",
        "num_sequences": NUM_SEQUENCES,
        "max_seq_length": MAX_SEQ_LENGTH,
        "reservoir": NUC_RESERVOIR_SIZE,
        "thresholds": ACTIVATION_THRESHOLDS,
    }
)
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs" / "structure_analysis" / f"multilayer_{ANALYSIS_KEY}"
CACHE_DIR = PROJECT_ROOT / ".cache" / "analysis" / "structure_analysis" / f"multilayer_{ANALYSIS_KEY}"
FIGURE_DIR = OUTPUT_ROOT / "figures"
for directory in (OUTPUT_ROOT, CACHE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

set_publication_style()
warnings.filterwarnings("ignore", category=FutureWarning)

layer_bundles, layer_catalog = load_layer_catalog(LAYER_PATHS)
display(layer_catalog.round(6))
print(f"Runtime target: {RUNTIME_TARGET}")
print(f"Device: {DEVICE} (preference: {DEVICE_PREFERENCE})")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Cache-only mode: {REQUIRE_CACHE}")

## Load The bpRNA-90 Subset

This cell creates a stable subset for the comparative analysis so that every layer sees the same nucleotide pool.


In [ ]:
ARCHIVED_DATASET = ARCHIVE_ROOT / "data" / "bprna" / f"bprna_subset_{NUM_SEQUENCES}_{MAX_SEQ_LENGTH}.pkl"
DATASET_CACHE = ARCHIVED_DATASET if ARCHIVED_DATASET.exists() else CACHE_DIR / ARCHIVED_DATASET.name

if USE_CACHE and DATASET_CACHE.exists():
    df_data = pd.read_pickle(DATASET_CACHE)
elif REQUIRE_CACHE:
    raise FileNotFoundError(f"Expected cached bpRNA subset at {DATASET_CACHE}.")
else:
    df_data = stream_bprna_dataframe(
        BPRNA_DATASET_ID,
        split=BPRNA_SPLIT,
        num_sequences=NUM_SEQUENCES,
    )
    df_data = df_data.copy()
    df_data["sequence"] = df_data["sequence"].str.slice(0, MAX_SEQ_LENGTH - 2)
    df_data["structural_annotation"] = [
        annotation[: len(sequence)]
        for sequence, annotation in zip(df_data["sequence"], df_data["structural_annotation"])
]
    df_data.to_pickle(DATASET_CACHE)

df_data = df_data.loc[:, ["sequence", "structural_annotation"]].copy().reset_index(drop=True)

structure_distribution_df = summarize_structure_distribution(df_data)
structure_distribution_df = structure_distribution_df[structure_distribution_df["structure"] != "K"].copy()
structure_distribution_df["fraction"] = structure_distribution_df["count"] / structure_distribution_df["count"].sum()
display(structure_distribution_df)
print(f"Loaded {len(df_data):,} sequences")

## Extraction and Cache Helpers

The multi-layer loop below saves one cache bundle per layer so repeated figure tuning does not force re-extraction.


In [ ]:
def structure_cache_paths(layer_index: int) -> dict[str, Path]:
    layer_dir = CACHE_DIR / f"layer_{layer_index:02d}"
    layer_dir.mkdir(parents=True, exist_ok=True)
    return {
        "stats": layer_dir / "structure_stats.npz",
        "results": layer_dir / "feature_structure_alignment.csv",
        "summary": layer_dir / "layer_summary.json",
    }


def save_structure_layer(layer_index: int, payload: dict, df_results: pd.DataFrame) -> None:
    cache_paths = structure_cache_paths(layer_index)
    np_payload = {
        "structure_chars": np.array(payload["structure_chars"], dtype=object),
        "feature_struct_counts": payload["feature_struct_counts"],
        "global_struct_counts": payload["global_struct_counts"],
        "feature_seq_counts": payload["feature_seq_counts"],
        "threshold_values": payload["threshold_values"],
        "reservoir_sae": payload["reservoir_sae"],
        "reservoir_hidden": payload["reservoir_hidden"],
        "reservoir_labels": payload["reservoir_labels"],
    }
    for threshold_index, threshold_counts in enumerate(payload["threshold_feature_struct_counts"]):
        np_payload[f"threshold_feature_struct_counts_{threshold_index}"] = threshold_counts
    np.savez_compressed(cache_paths["stats"], **np_payload)
    df_results.to_csv(cache_paths["results"], index=False)
    cache_paths["summary"].write_text(
        json.dumps(
            {
                "n_processed": int(payload["n_processed"]),
                "n_skipped": int(payload["n_skipped"]),
                "total_nucleotides": int(payload["total_nucleotides"]),
            },
            indent=2,
        )
    )


def load_structure_layer(
    layer_index: int,
    include_threshold_counts: bool = False,
    include_reservoir: bool = False,
) -> tuple[dict, pd.DataFrame]:
    cache_paths = structure_cache_paths(layer_index)
    with np.load(cache_paths["stats"], allow_pickle=True) as packed:
        payload = {
            "structure_chars": packed["structure_chars"].tolist(),
            "feature_struct_counts": packed["feature_struct_counts"],
            "global_struct_counts": packed["global_struct_counts"],
            "feature_seq_counts": packed["feature_seq_counts"],
        }
        if include_threshold_counts and "threshold_values" in packed.files:
            threshold_values = packed["threshold_values"]
            payload["threshold_values"] = threshold_values
            payload["threshold_feature_struct_counts"] = [
                packed[f"threshold_feature_struct_counts_{threshold_index}"]
                for threshold_index in range(len(threshold_values))
            ]
        if include_reservoir:
            payload["reservoir_sae"] = packed["reservoir_sae"]
            payload["reservoir_hidden"] = packed["reservoir_hidden"]
            payload["reservoir_labels"] = packed["reservoir_labels"]
    df_results = pd.read_csv(cache_paths["results"])
    return payload, df_results


def filter_structure_payload(
    payload: dict,
    excluded_structures: set[str],
    include_threshold_counts: bool = False,
    include_reservoir: bool = False,
) -> dict:
    structure_order = [
        structure_label
        for structure_label in payload["structure_chars"]
        if structure_label not in excluded_structures
    ]
    keep_indices = [payload["structure_chars"].index(label) for label in structure_order]
    filtered_payload = {
        "structure_chars": structure_order,
        "feature_struct_counts": payload["feature_struct_counts"][:, keep_indices],
        "global_struct_counts": payload["global_struct_counts"][keep_indices],
        "feature_seq_counts": payload["feature_seq_counts"],
    }
    if include_threshold_counts and "threshold_feature_struct_counts" in payload:
        filtered_payload["threshold_values"] = payload["threshold_values"]
        filtered_payload["threshold_feature_struct_counts"] = [
            threshold_counts[:, keep_indices]
            for threshold_counts in payload["threshold_feature_struct_counts"]
        ]
    if include_reservoir and "reservoir_labels" in payload:
        remap = np.full(len(payload["structure_chars"]), -1, dtype=int)
        remap[keep_indices] = np.arange(len(keep_indices))
        keep_mask = np.isin(payload["reservoir_labels"], keep_indices)
        filtered_payload["reservoir_hidden"] = payload["reservoir_hidden"][keep_mask]
        filtered_payload["reservoir_sae"] = payload["reservoir_sae"][keep_mask]
        filtered_payload["reservoir_labels"] = remap[payload["reservoir_labels"][keep_mask]].astype(np.int16)
    return filtered_payload


def summarize_structure_layer(layer_index: int, df_results: pd.DataFrame) -> dict:
    if df_results.empty:
        return {
            "layer_index": layer_index,
            "n_features": 0,
            "n_significant": 0,
            "median_selectivity": np.nan,
            "max_selectivity": np.nan,
            "mean_enrichment": np.nan,
        }
    significant_mask = df_results["significant_bonf"] if "significant_bonf" in df_results else pd.Series(False, index=df_results.index)
    return {
        "layer_index": layer_index,
        "n_features": int(len(df_results)),
        "n_significant": int(significant_mask.sum()),
        "median_selectivity": float(df_results["selectivity"].median()),
        "max_selectivity": float(df_results["selectivity"].max()),
        "mean_enrichment": float(df_results["enrichment_ratio"].mean()),
    }

## Compute Or Load Per-Layer Structure Statistics

Each layer is processed independently, but every layer sees the same dataset subset and the same threshold sweep.


In [ ]:
EXCLUDED_STRUCTURES = {"K"}
structure_layer_metadata = {}
structure_feature_tables = {}
structure_summary_rows = []

for layer_index, bundle in layer_bundles.items():
    cache_paths = structure_cache_paths(layer_index)
    should_use_cache = USE_CACHE and cache_paths["stats"].exists() and cache_paths["results"].exists()
    if should_use_cache:
        payload, _ = load_structure_layer(layer_index)
        should_use_cache = int(payload["global_struct_counts"].sum()) > 0

    if not should_use_cache and REQUIRE_CACHE:
        raise FileNotFoundError(
            "Cached biological alignment artifacts are required but missing for "
            f"layer {layer_index}: {cache_paths['stats']} and {cache_paths['results']}"
        )

    if not should_use_cache:
        embedder, sae, act_mean, act_std = load_layer_model(
            bundle,
            device=DEVICE,
            quantization=MODEL_QUANTIZATION,
        )[1:]
        payload = collect_structure_alignment(
            df_data,
            embedder,
            sae,
            act_mean=act_mean,
            act_std=act_std,
            max_seq_length=MAX_SEQ_LENGTH,
            reservoir_size=NUC_RESERVOIR_SIZE,
            activation_thresholds=ACTIVATION_THRESHOLDS,
        )
        raw_df_results = compute_structure_feature_statistics(
            payload["feature_struct_counts"],
            payload["global_struct_counts"],
            payload["feature_seq_counts"],
            structure_chars=payload["structure_chars"],
            min_sequence_count=MIN_SEQUENCE_COUNT,
        )
        save_structure_layer(layer_index, payload, raw_df_results)
        safe_unload(embedder, sae)
        del raw_df_results

    filtered_payload = filter_structure_payload(
        payload,
        excluded_structures=EXCLUDED_STRUCTURES,
    )
    df_results = compute_structure_feature_statistics(
        filtered_payload["feature_struct_counts"],
        filtered_payload["global_struct_counts"],
        filtered_payload["feature_seq_counts"],
        structure_chars=filtered_payload["structure_chars"],
        min_sequence_count=MIN_SEQUENCE_COUNT,
    )

    structure_layer_metadata[layer_index] = {
        "structure_chars": filtered_payload["structure_chars"],
        "global_struct_counts": filtered_payload["global_struct_counts"].astype(np.int64, copy=False),
    }
    structure_feature_tables[layer_index] = df_results
    structure_summary_rows.append(summarize_structure_layer(layer_index, df_results))

    del payload, filtered_payload
    gc.collect()

structure_summary_df = pd.DataFrame(structure_summary_rows).sort_values("layer_index").reset_index(drop=True)
display(structure_summary_df.round(4))

In [ ]:
STRUCTURE_ORDER = structure_layer_metadata[REFERENCE_LAYER]["structure_chars"]
STRUCTURE_NAME_MAP = {label: STRUCTURE_LABELS[label] for label in STRUCTURE_ORDER}
PROFILE_COLUMNS = [f"frac_{label}" for label in STRUCTURE_ORDER]

background_counts = structure_layer_metadata[REFERENCE_LAYER]["global_struct_counts"]
structure_distribution_df = pd.DataFrame(
    {
        "structure": STRUCTURE_ORDER,
        "structure_name": [STRUCTURE_LABELS[label] for label in STRUCTURE_ORDER],
        "count": background_counts.astype(int),
    }
)
structure_distribution_df["fraction"] = structure_distribution_df["count"] / structure_distribution_df["count"].sum()


def balanced_label_sample(labels: np.ndarray, sample_size: int, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    unique_labels = np.unique(labels)
    if len(unique_labels) == 0:
        return np.array([], dtype=int)

    per_group = max(sample_size // len(unique_labels), 1)
    chosen = []
    for label in unique_labels:
        label_indices = np.where(labels == label)[0]
        take = min(len(label_indices), per_group)
        chosen.extend(rng.choice(label_indices, size=take, replace=False).tolist())

    chosen = np.array(sorted(set(chosen)), dtype=int)
    target_size = min(sample_size, len(labels))
    if len(chosen) < target_size:
        remaining = np.setdiff1d(np.arange(len(labels)), chosen, assume_unique=False)
        extra = min(target_size - len(chosen), len(remaining))
        if extra > 0:
            chosen = np.concatenate([chosen, rng.choice(remaining, size=extra, replace=False)])
    if len(chosen) > target_size:
        chosen = rng.choice(chosen, size=target_size, replace=False)
    return np.sort(chosen)


def projection_metrics(projection: np.ndarray, labels: np.ndarray, seed: int = 42) -> dict[str, float]:
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(projection) <= len(unique_labels):
        return {"silhouette": np.nan, "ari": np.nan}

    try:
        silhouette = silhouette_score(projection, labels)
    except ValueError:
        silhouette = np.nan
    clusters = KMeans(n_clusters=len(unique_labels), n_init=10, random_state=seed).fit_predict(projection)
    ari = adjusted_rand_score(labels, clusters)
    return {"silhouette": float(silhouette), "ari": float(ari)}


def build_structure_tracks(reference_layer: int = REFERENCE_LAYER) -> list[dict]:
    reference_df = structure_feature_tables[reference_layer]
    tracks = []
    for structure_label in STRUCTURE_ORDER:
        candidates = reference_df[reference_df["preferred_structure"] == structure_label]
        if candidates.empty:
            continue

        anchor_row = candidates.iloc[0]
        anchor_profile = anchor_row[PROFILE_COLUMNS].to_numpy(dtype=float)
        anchor_norm = np.linalg.norm(anchor_profile) + 1e-12
        layer_matches = {}

        for layer_index, df_layer in structure_feature_tables.items():
            layer_candidates = df_layer[df_layer["preferred_structure"] == structure_label]
            if layer_candidates.empty:
                layer_matches[layer_index] = None
                continue

            profiles = layer_candidates[PROFILE_COLUMNS].to_numpy(dtype=float)
            profile_norms = np.linalg.norm(profiles, axis=1) + 1e-12
            similarities = (profiles @ anchor_profile) / (profile_norms * anchor_norm)
            best_position = int(np.argmax(similarities))
            best_row = layer_candidates.iloc[best_position]
            layer_matches[layer_index] = {
                "row": best_row,
                "similarity_to_anchor": float(similarities[best_position]),
            }

        tracks.append(
            {
                "structure": structure_label,
                "track_label": STRUCTURE_LABELS[structure_label],
                "reference_layer": int(reference_layer),
                "reference_feature_idx": int(anchor_row["feature_idx"]),
                "layers": layer_matches,
            }
        )
    return tracks


def load_structure_projection_sample(
    layer_index: int,
    sample_size: int,
    seed: int = PCA_RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    payload, _ = load_structure_layer(layer_index, include_reservoir=True)
    filtered_payload = filter_structure_payload(
        payload,
        excluded_structures=EXCLUDED_STRUCTURES,
        include_reservoir=True,
    )
    labels = np.array(
        [filtered_payload["structure_chars"][label_index] for label_index in filtered_payload["reservoir_labels"]]
    )
    sample_indices = balanced_label_sample(labels, min(sample_size, len(labels)), seed=seed)
    hidden_sample = filtered_payload["reservoir_hidden"][sample_indices].astype(np.float32)
    sae_sample = filtered_payload["reservoir_sae"][sample_indices].astype(np.float32)
    label_sample = labels[sample_indices]
    del payload, filtered_payload, labels
    gc.collect()
    return hidden_sample, sae_sample, label_sample


structure_tracks = build_structure_tracks()

## Plot 1: Background Structure Distribution

This figure sets the base rates that every selectivity and enrichment metric is compared against.


In [ ]:
fig, ax = plt.subplots(figsize=(9.6, 5.4))
plot_df = structure_distribution_df.sort_values("count", ascending=False).copy()
plot_df["label"] = plot_df["structure"] + "  " + plot_df["structure_name"]

bars = ax.bar(
    plot_df["label"],
    plot_df["count"],
    color=[STRUCTURE_PALETTE.get(label, "#9ca3af") for label in plot_df["structure"]],
    edgecolor="white",
    linewidth=0.8,
    zorder=3,
)
ax.set_ylabel("Nucleotide count")
beautify_axes(ax, rotate_x=28)
for bar, count in zip(bars, plot_df["count"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )
fig.tight_layout()
show_and_close(fig, "01_background_structure_distribution")

## Plot 2: Selectivity Distribution By Layer

This keeps the old histogram style, but shows one layer at a time so the high-selectivity tail stays readable.

In [ ]:
chance_level = 1 / len(STRUCTURE_ORDER)
bins = np.linspace(chance_level * 0.75, 1.0, 46)

for layer_index in sorted(layer_bundles):
    df_layer = structure_feature_tables[layer_index]
    selectivity_values = df_layer["selectivity"].to_numpy()
    median_value = float(np.median(selectivity_values))
    tail_start = min(max(float(np.quantile(selectivity_values, 0.97)), chance_level + 0.22), 0.95)
    tail_values = selectivity_values[selectivity_values >= tail_start]
    top_tail_values = np.sort(tail_values)[-3:] if len(tail_values) > 0 else np.array([])

    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.hist(
        selectivity_values,
        bins=bins,
        color=LAYER_COLORS[layer_index],
        edgecolor="white",
        alpha=0.88,
        zorder=3,
    )
    ax.axvline(chance_level, color="#7c7c7c", linestyle=":", linewidth=1.8, zorder=4)
    ax.axvline(median_value, color="#c1121f", linestyle="--", linewidth=2.1, zorder=4)
    ax.axvspan(tail_start, 1.0, color="#f59e0b", alpha=0.08, zorder=1)

    summary_lines = [
        f"Median: {median_value:.3f}",
        f"Uniform-class reference: {chance_level:.3f}",
        f"Tail >= {tail_start:.3f}: {len(tail_values)} features",
    ]
    if len(top_tail_values) > 0:
        summary_lines.append(
            "Top tail values: " + ", ".join(f"{value:.3f}" for value in top_tail_values)
        )
    ax.text(
        0.98,
        0.97,
        "\n".join(summary_lines),
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=12,
        fontweight="bold",
        bbox={
            "boxstyle": "round,pad=0.35",
            "facecolor": "white",
            "edgecolor": "#d1d5db",
            "alpha": 0.92,
        },
    )

    ax.set_xlim(chance_level * 0.75, 1.0)
    ax.set_xlabel("Selectivity")
    ax.set_ylabel("Number of features")
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")
    fig.tight_layout()
    show_and_close(fig, f"02_selectivity_distribution_layer_{layer_index:02d}")

## Plot 3: Structure Preference Counts

This heatmap keeps the structure preference counts view for the filtered structure set.

In [ ]:
preferred_counts_rows = []
for layer_index, df_results in structure_feature_tables.items():
    sig_df = df_results[df_results["significant_bonf"]].copy()
    counts = sig_df["preferred_structure"].value_counts()
    for structure_label in STRUCTURE_ORDER:
        preferred_counts_rows.append(
            {
                "layer_index": layer_index,
                "structure": structure_label,
                "count": int(counts.get(structure_label, 0)),
            }
        )

preferred_counts_df = pd.DataFrame(preferred_counts_rows)
preferred_heatmap = preferred_counts_df.pivot(index="structure", columns="layer_index", values="count").fillna(0).astype(int)
preferred_heatmap = preferred_heatmap.loc[STRUCTURE_ORDER]

fig, ax = plt.subplots(figsize=(8.2, 5.6))
sns.heatmap(
    preferred_heatmap,
    annot=True,
    fmt="d",
    cmap=sns.light_palette("#0f766e", as_cmap=True),
    annot_kws={"fontsize": 12, "fontweight": "bold"},
    cbar_kws={"label": "Bonferroni-significant feature count"},
    ax=ax,
)
ax.set_xlabel("Layer index")
ax.set_ylabel("Preferred structure")
ax.set_yticklabels([f"{label}  {STRUCTURE_LABELS[label]}" for label in preferred_heatmap.index], rotation=0)
beautify_axes(ax)
for label in ax.collections[0].colorbar.ax.get_yticklabels():
    label.set_fontweight("bold")
ax.collections[0].colorbar.ax.tick_params(labelsize=12)
fig.tight_layout()
show_and_close(fig, "03_structure_preference_counts")

## Plot 4: Layer-Wise Top 10 Selective Features

For each layer, this keeps the current view of the strongest structure-selective SAE features.

In [ ]:
for layer_index in sorted(layer_bundles):
    df_layer = structure_feature_tables[layer_index]
    significant_df = df_layer[df_layer["significant_bonf"]].copy()
    if significant_df.empty:
        significant_df = df_layer.copy()
    df_layer = significant_df.head(TOP_FEATURES_PER_LAYER)
    y_positions = np.arange(len(df_layer))
    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.barh(
        y_positions,
        df_layer["selectivity"],
        color=[STRUCTURE_PALETTE.get(label, "#9ca3af") for label in df_layer["preferred_structure"]],
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels([f"F{feature_idx}" for feature_idx in df_layer["feature_idx"]])
    ax.invert_yaxis()
    ax.set_xlabel("Selectivity")
    ax.set_ylabel("Feature")
    beautify_axes(ax)
    present_structures = list(dict.fromkeys(df_layer["preferred_structure"]))
    legend = fig.legend(
        [plt.Line2D([], [], marker="s", linestyle="", markersize=9,
                    markerfacecolor=STRUCTURE_PALETTE.get(label, "#9ca3af"),
                    markeredgecolor="white") for label in present_structures],
        [f"{label} — {STRUCTURE_LABELS[label]}" for label in present_structures],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.985),
        ncol=min(3, len(present_structures)),
        title=f"Layer {layer_index} · Preferred structure",
        frameon=True,
    )
    legend.get_title().set_fontweight("bold")
    for text in legend.get_texts():
        text.set_fontweight("bold")
    for y_position, (_, row) in zip(y_positions, df_layer.iterrows()):
        ax.text(
            row["selectivity"] + 0.01,
            y_position,
            f"{row['preferred_structure']}  {row['enrichment_ratio']:.1f}x",
            va="center",
            fontsize=12,
            fontweight="bold",
        )

    fig.tight_layout(rect=(0, 0, 1, 0.88))
    show_and_close(fig, f"04_top_structure_features_layer_{layer_index:02d}")

## Plot 5: Layer-Wise Top 10 Enrichment Ratios

This follows the old notebook style, but compares the strongest enrichment ratios for all three layers side by side.

In [ ]:
for layer_index in sorted(layer_bundles):
    df_layer = structure_feature_tables[layer_index]
    significant_df = df_layer[df_layer["significant_bonf"]].copy()
    if significant_df.empty:
        significant_df = df_layer.copy()
    df_layer = significant_df.sort_values(["enrichment_ratio", "selectivity"], ascending=[False, False]).head(TOP_FEATURES_PER_LAYER)

    y_positions = np.arange(len(df_layer))
    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.barh(
        y_positions,
        df_layer["enrichment_ratio"],
        color=[STRUCTURE_PALETTE.get(label, "#9ca3af") for label in df_layer["preferred_structure"]],
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels([f"F{feature_idx}" for feature_idx in df_layer["feature_idx"]])
    ax.invert_yaxis()
    ax.set_xlabel("Enrichment ratio")
    ax.set_ylabel("Feature")
    beautify_axes(ax)
    present_structures = list(dict.fromkeys(df_layer["preferred_structure"]))
    legend = fig.legend(
        [plt.Line2D([], [], marker="s", linestyle="", markersize=9,
                    markerfacecolor=STRUCTURE_PALETTE.get(label, "#9ca3af"),
                    markeredgecolor="white") for label in present_structures],
        [f"{label} — {STRUCTURE_LABELS[label]}" for label in present_structures],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.985),
        ncol=min(3, len(present_structures)),
        title=f"Layer {layer_index} · Preferred structure",
        frameon=True,
    )
    legend.get_title().set_fontweight("bold")
    for text in legend.get_texts():
        text.set_fontweight("bold")
    for y_position, (_, row) in zip(y_positions, df_layer.iterrows()):
        ax.text(
            row["enrichment_ratio"] + 0.05,
            y_position,
            f"{row['preferred_structure']}  sel={row['selectivity']:.2f}",
            va="center",
            fontsize=12,
            fontweight="bold",
        )

    fig.tight_layout(rect=(0, 0, 1, 0.88))
    show_and_close(fig, f"05_top_structure_enrichment_layer_{layer_index:02d}")

## Plot 6: Representative Structure-Composition Profiles

Each column is anchored on one reference-layer feature. The other layers show the closest profile-matched feature for that same structure track, so the feature ids are expected to differ across layers.

In [ ]:
x_positions = np.arange(len(structure_tracks))
for layer_index in sorted(layer_bundles):
    track_matches = [track["layers"].get(layer_index) for track in structure_tracks]
    track_rows = [match["row"] if match is not None else None for match in track_matches]
    bottoms = np.zeros(len(track_rows))

    fig, ax = plt.subplots(figsize=(13.2, 4.8))
    for structure_label in STRUCTURE_ORDER:
        values = np.array(
            [
                row[f"frac_{structure_label}"] if row is not None else 0.0
                for row in track_rows
            ]
        )
        ax.bar(
            x_positions,
            values,
            bottom=bottoms,
            color=STRUCTURE_PALETTE.get(structure_label, "#9ca3af"),
            edgecolor="white",
            linewidth=0.5,
            label=f"{structure_label}  {STRUCTURE_LABELS[structure_label]}",
            zorder=3,
        )
        bottoms += values

    for x_position, track, match in zip(x_positions, structure_tracks, track_matches):
        if match is None:
            label = "no match"
        elif layer_index == track["reference_layer"]:
            label = f"anchor F{int(match['row']['feature_idx'])}"
        else:
            label = (
                f"best match F{int(match['row']['feature_idx'])}\n"
                f"cos={match['similarity_to_anchor']:.2f}"
            )
        ax.text(
            x_position,
            1.02,
            label,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
        )

    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Composition")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [
            f"{track['structure']}\nanchor L{track['reference_layer']} F{track['reference_feature_idx']}"
            for track in structure_tracks
        ]
    )
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")

    legend = ax.legend(
        loc="lower center", bbox_to_anchor=(0.5, 1.16), ncol=4,
        title="Structure class", frameon=True, borderaxespad=0.0,
    )
    legend.get_title().set_fontweight("bold")
    for text in legend.get_texts():
        text.set_fontweight("bold")

    fig.tight_layout(rect=(0, 0, 1, 0.84))
    show_and_close(fig, f"06_structure_composition_layer_{layer_index:02d}")

## Plot 7: PCA Of Raw Hidden States Versus SAE Activations

This compares all three layers with the old family notebook styling and reports silhouette and ARI for a more meaningful geometric comparison.

In [ ]:
projection_rows = []
layer_projection_results = {}

for layer_index in sorted(layer_bundles):
    hidden_sample, sae_sample, label_sample = load_structure_projection_sample(
        layer_index,
        sample_size=PCA_SAMPLE_SIZE,
        seed=PCA_RANDOM_STATE,
    )

    active_feature_mask = (sae_sample > 0.05).any(axis=0)
    sae_projection_input = sae_sample[:, active_feature_mask]
    if sae_projection_input.shape[1] > PCA_ACTIVE_FEATURE_CAP:
        variances = sae_projection_input.var(axis=0)
        top_feature_indices = np.argsort(variances)[-PCA_ACTIVE_FEATURE_CAP:]
        sae_projection_input = sae_projection_input[:, top_feature_indices]

    hidden_model = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
    sae_model = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
    hidden_projection = hidden_model.fit_transform(hidden_sample)
    sae_projection = sae_model.fit_transform(sae_projection_input)

    layer_projection_results[layer_index] = {
        "labels": label_sample,
        "BiRNA-BERT": {
            "projection": hidden_projection,
            "explained": float(hidden_model.explained_variance_ratio_[:2].sum()),
            "metrics": projection_metrics(hidden_projection, label_sample, seed=PCA_RANDOM_STATE),
        },
        "SAE": {
            "projection": sae_projection,
            "explained": float(sae_model.explained_variance_ratio_[:2].sum()),
            "metrics": projection_metrics(sae_projection, label_sample, seed=PCA_RANDOM_STATE),
        },
    }

    for representation_name in ["BiRNA-BERT", "SAE"]:
        projection_bundle = layer_projection_results[layer_index][representation_name]
        projection_rows.append(
            {
                "layer_index": layer_index,
                "representation": representation_name,
                "explained_variance": projection_bundle["explained"],
                "silhouette": projection_bundle["metrics"]["silhouette"],
                "ari": projection_bundle["metrics"]["ari"],
            }
        )

    del hidden_sample, sae_sample, sae_projection_input, active_feature_mask
    gc.collect()

projection_metrics_df = pd.DataFrame(projection_rows)
display(projection_metrics_df.round(4))
print("Metric guide: higher is better for explained variance, silhouette, and ARI.")

for layer_index in sorted(layer_bundles):
    label_sample = layer_projection_results[layer_index]["labels"]
    for representation_name in ["BiRNA-BERT", "SAE"]:
        fig, ax = plt.subplots(figsize=(7.2, 6.6))
        projection_bundle = layer_projection_results[layer_index][representation_name]
        projection = projection_bundle["projection"]
        metrics = projection_bundle["metrics"]

        for structure_label in STRUCTURE_ORDER:
            mask = label_sample == structure_label
            if not mask.any():
                continue
            ax.scatter(
                projection[mask, 0],
                projection[mask, 1],
                s=18,
                alpha=0.68,
                color=STRUCTURE_PALETTE.get(structure_label, "#9ca3af"),
                edgecolors="white",
                linewidths=0.2,
                rasterized=True,
            )


        legend = fig.legend(
            [plt.Line2D([], [], marker="o", linestyle="", markersize=8,
                        markerfacecolor=STRUCTURE_PALETTE.get(label, "#9ca3af"),
                        markeredgecolor="white") for label in STRUCTURE_ORDER],
            [f"{label} {STRUCTURE_LABELS[label]}" for label in STRUCTURE_ORDER],
            loc="upper center", bbox_to_anchor=(0.5, 0.985), ncol=4,
            title=f"Structure class · Layer {layer_index} · {representation_name}", frameon=True, columnspacing=1.0,
            handletextpad=0.4, borderaxespad=0.0,
        )
        legend.get_title().set_fontweight("bold")
        for text in legend.get_texts():
            text.set_fontweight("bold")

        metric_text = (
            f"Explained {projection_bundle['explained'] * 100:.1f}%  ·  "
            f"Silhouette {metrics['silhouette']:.3f}  ·  ARI {metrics['ari']:.3f}"
        )
        fig.text(
            0.5,
            0.015,
            metric_text,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color="#374151",
        )
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        beautify_axes(ax)
        fig.tight_layout(rect=(0, 0.075, 1, 0.82))
        representation_slug = representation_name.lower().replace(" ", "_").replace("-", "_")
        show_and_close(fig, f"07_pca_{representation_slug}_layer_{layer_index:02d}")

## Cleanup

All extraction loops unload models after use, but this cell is here if you want to force an extra cleanup pass before continuing.


In [ ]:
for name in [
    "projection_metrics_df",
    "layer_projection_results",
]:
    globals().pop(name, None)
plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Notebook state is clean.")
